Save as: module2-simulation-lab.ipynb

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moncap/micro-course/blob/main/module-02/module2-simulation-lab.ipynb)

# Lab 2 — Silicon Consumers: Demand, Framing, and a GARP Audit
*Microeconomic Theory for Experimentalists & Applied Microeconomists*

**Research question:** Do LLM consumers produce downward-sloping demand,
respond to a pure framing manipulation (a "sale" at identical final
prices), and — the audit — are their choices *consistent* in the
revealed-preference sense, at human-like rates?

**The task:** allocate a \$20 budget between coffee and pastries on four
price menus (the same four the class answered in Lecture 1):

| Menu | Coffee | Pastry |
|---|---|---|
| m1 | \$4 | \$2 |
| m2 | \$2 | \$4 |
| m3 | \$5 | \$5 |
| m4 | \$2 | \$2 |

Each agent answers **all four menus in one prompt**, so consistency across
menus is a property of one agent — exactly as in a human within-subject
design.

**Human benchmarks:** Andreoni & Miller (2002): roughly 90% of subjects
GARP-consistent. Choi et al. (2007): high but imperfect consistency, with
heterogeneity. Downward-sloping demand. Plus the class's own Lecture-1
choices for a three-way comparison.

**Hypotheses to state before running (see handout):** H1 demand slopes
down across menus; H2 "sale" framing raises coffee purchases at identical
final prices; H3 the per-agent WARP violation rate is in the human
ballpark (~10%) — **both directions of failure are findings**: too many
violations means the agents can't do budgets; zero violations means they
don't reproduce the human error process (PS2 Part C takes this head-on).

**How to run:** no prior Python experience needed. Run the cells top to
bottom. Your required modification (handout) goes **only** in the cells
marked **CHANGE HERE**.


## Setup and the DRY_RUN switch

The notebook starts in **DRY_RUN mode**: it executes end-to-end on synthetic
placeholder data, with no API key and no cost, so you can check your design
and see the analysis pipeline work.

> ⚠️ **DRY_RUN output is fake.** It is generated by `mock_response()`, not
> by a model. Never report it as a result.

When your design is ready, set `DRY_RUN = False` in the cell below and run
again in Colab — you will be asked for your Anthropic API key via a hidden
prompt (the key is never stored in the notebook file).


In [ ]:
%pip install anthropic pandas matplotlib --quiet

In [ ]:
import itertools
import json
import random
import re
import time

import pandas as pd

DRY_RUN = True   # <-- set to False in Colab to query the real model

MODEL = "claude-sonnet-5"   # swap models here to test robustness across models
TEMPERATURE = 1.0            # fixed and reported — part of the experimental design
N_PER_CELL = 30              # decisions per persona x treatment cell

if DRY_RUN:
    print("=" * 62)
    print("DRY RUN: synthetic placeholder data — NOT model behavior.")
    print("Set DRY_RUN = False to run the real experiment.")
    print("=" * 62)

    class _MockBlock:
        def __init__(self, text):
            self.text = text

    class _MockResponse:
        def __init__(self, text):
            self.content = [_MockBlock(text)]

    class _MockMessages:
        def create(self, **kwargs):
            # mock_response() is defined in the design cell below
            return _MockResponse(mock_response(kwargs["messages"][0]["content"]))

    class _MockClient:
        messages = _MockMessages()

    client = _MockClient()
else:
    from getpass import getpass

    import anthropic

    # Never hard-code API keys. getpass keeps the key out of the notebook file.
    API_KEY = getpass("Paste your Anthropic API key: ")
    client = anthropic.Anthropic(api_key=API_KEY)


## Experimental design — CHANGE HERE (all four blocks live in this cell)

3 personas × 2 framings = 6 cells × 30 agents = 180 calls, each returning
four bundles.

Personas are in **plain behavioral language** — no utility functions. If we
wrote "you have Cobb–Douglas preferences with equal shares", the model
would compute the textbook answer; that tests arithmetic, not behavior.

The framing treatment changes **wording only**: in the `sale` condition,
coffee's price is described as "half its usual price," but the final
prices are identical to the `plain` condition. Standard theory says the
framing cannot matter; reference-dependent consumers (and marketing
departments) disagree.

Your required modification will usually mean **(1)** adding a persona or a
menu (`MENUS`), **(2)** weaving it into `build_prompt` *without changing
any other wording across conditions*, and **(3)** touching
`parse_response` / `mock_response` only if the answer format changes.


In [ ]:
PERSONAS = {                                     # CHANGE HERE (1 of 4)
    "baseline": (
        "You are an adult doing your regular midweek shopping at a cafe."
    ),
    "variety_lover": (
        "You are an adult doing your regular midweek shopping at a cafe. "
        "You get bored consuming too much of the same thing and like to "
        "mix your purchases."
    ),
    "coffee_devotee": (
        "You are an adult doing your regular midweek shopping at a cafe. "
        "You deeply love coffee; it is the highlight of your day, and "
        "pastries are an afterthought."
    ),
}

PERSONA_GRID = {
    "persona": list(PERSONAS),
    "framing": ["plain", "sale"],   # identical final prices, different wording
}

BUDGET = 20
MENUS = {                            # (coffee price, pastry price) per menu
    "m1": (4, 2),
    "m2": (2, 4),
    "m3": (5, 5),
    "m4": (2, 2),
}


def build_prompt(persona: dict) -> str:          # CHANGE HERE (2 of 4)
    """Four budget menus, one prompt. Design rules: persona first, task
    second, response format last; identical wording across conditions
    except the framing sentence; no hint that any allocation is 'rational'
    or 'balanced'."""

    def price_line(name, pc, pp):
        if persona["framing"] == "sale":
            coffee_txt = (f"coffee is on sale at half its usual price: "
                          f"${pc} per cup (usually ${2 * pc})")
        else:
            coffee_txt = f"coffee costs ${pc} per cup"
        return (f"  Menu {name}: {coffee_txt}; pastries cost ${pp} each.")

    menu_lines = "\n".join(
        price_line(name, pc, pp) for name, (pc, pp) in MENUS.items())

    return (
        f"{PERSONAS[persona['persona']]}\n\n"
        f"On four separate days you have exactly ${BUDGET} to spend at the "
        "cafe, under four different price menus:\n\n"
        f"{menu_lines}\n\n"
        "For each menu, decide how many cups of coffee and how many "
        "pastries you buy (whole numbers; you cannot exceed the budget; "
        "you need not spend it all). Decide as you genuinely would.\n\n"
        'Respond with ONLY a JSON object like {"m1": [cups, pastries], '
        '"m2": [cups, pastries], "m3": [cups, pastries], '
        '"m4": [cups, pastries]}.'
    )


def parse_response(text: str) -> "dict | None":  # CHANGE HERE (3 of 4)
    """Extract four bundles. Return None on failure — never guess.
    Validates: every menu present, whole-number quantities, within budget."""
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        data = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None
    bundles = {}
    for name, (pc, pp) in MENUS.items():
        b = data.get(name)
        if (not isinstance(b, list) or len(b) != 2
                or not all(isinstance(q, int) and q >= 0 for q in b)
                or pc * b[0] + pp * b[1] > BUDGET):
            return None
        bundles[name] = (b[0], b[1])
    return bundles


def mock_response(prompt: str) -> str:           # CHANGE HERE (4 of 4)
    """DRY_RUN only: synthetic placeholder answers so the notebook executes
    without API calls. NOT model behavior — these numbers are made up and
    deliberately produce legible patterns (downward demand, a framing
    bump, occasional inconsistency). Edit only if the answer format
    changes."""
    if "highlight of your day" in prompt:        # coffee devotee
        share = 0.75
    elif "bored consuming too much" in prompt:   # variety lover
        share = 0.50
    else:                                        # baseline
        share = 0.55
    if "on sale at half its usual price" in prompt:
        share = min(share + 0.12, 0.95)          # framing bump
    out = {}
    for name, (pc, pp) in MENUS.items():
        s = min(max(share + random.gauss(0, 0.08), 0.05), 0.95)
        cups = int(s * BUDGET // pc)
        pastries = int((BUDGET - cups * pc) // pp)
        if random.random() < 0.5:                # don't always exhaust budget
            pastries = max(pastries - 1, 0)
        out[name] = [cups, pastries]
    return json.dumps(out)


## Run the experiment *(do not modify)*

Runs every persona × framing cell, logs parse failures instead of dropping
them, saves `lab2_results.csv` and `lab2_prompts_log.json` (required in
your write-up appendix). With `DRY_RUN = False` this makes ~180 API calls.


In [ ]:
def run_experiment() -> pd.DataFrame:
    keys = list(PERSONA_GRID)
    cells = [dict(zip(keys, combo)) for combo in itertools.product(*PERSONA_GRID.values())]
    print(f"{len(cells)} cells x {N_PER_CELL} agents = {len(cells) * N_PER_CELL} calls")

    rows = []
    for cell in cells:
        prompt = build_prompt(cell)
        for i in range(N_PER_CELL):
            try:
                response = client.messages.create(
                    model=MODEL,
                    max_tokens=200,
                    temperature=TEMPERATURE,
                    messages=[{"role": "user", "content": prompt}],
                )
                raw = response.content[0].text
                bundles = parse_response(raw)
            except Exception as err:            # log failures; never drop silently
                raw, bundles = f"ERROR: {err}", None
                time.sleep(5)
            row = {**cell, "rep": i, "raw": raw,
                   "model": MODEL, "temperature": TEMPERATURE}
            for name in MENUS:
                row[f"{name}_coffee"] = None if bundles is None else bundles[name][0]
                row[f"{name}_pastry"] = None if bundles is None else bundles[name][1]
            rows.append(row)
        done = [r for r in rows if r["m1_coffee"] is not None]
        print(f"  cell {cell} done ({len(done)}/{len(rows)} parsed)")

    df = pd.DataFrame(rows)
    df.to_csv("lab2_results.csv", index=False)
    # Save exact prompts — required in the write-up appendix.
    with open("lab2_prompts_log.json", "w") as f:
        json.dump({str(c): build_prompt(c) for c in cells}, f, indent=2)
    return df


df = run_experiment()


## Analysis vs. human benchmarks *(do not modify)*

Three outputs: (1) aggregate demand — mean coffee purchases by menu, to be
read against coffee's price; (2) the framing effect at identical final
prices; (3) the WARP audit — for each agent, every pair of menus is
checked: if the bundle chosen under menu *j* was affordable under menu *i*
(where a different bundle was chosen), the agent revealed the menu-*i*
bundle preferred; if the reverse also holds, that pair is a violation.

Human benchmarks: ~90% of Andreoni & Miller's subjects were
GARP-consistent; Choi et al. found high but imperfect consistency.


In [ ]:
valid = df.dropna(subset=[f"{m}_coffee" for m in MENUS]).copy()
print(f"Parse-failure rate: {df['m1_coffee'].isna().mean():.3f} "
      "(report this in Section 4 of the write-up)")

print("\n=== Aggregate demand: mean coffee purchases by menu ===")
for name, (pc, pp) in MENUS.items():
    print(f"  {name} (coffee ${pc}): {valid[f'{name}_coffee'].mean():.2f} cups")
print("Downward-sloping? Compare quantities where coffee is $2, $4, $5.")

print("\n=== Framing effect (identical final prices) ===")
frame_table = valid.groupby("framing")[[f"{m}_coffee" for m in MENUS]].mean().round(2)
print(frame_table)
print("Standard theory: rows should be identical. Reference dependence: "
      "'sale' row higher.")


def warp_violations(row) -> int:
    """Count violating menu pairs for one agent's four bundles."""
    bundles = {m: (row[f"{m}_coffee"], row[f"{m}_pastry"]) for m in MENUS}
    cost = lambda prices, b: prices[0] * b[0] + prices[1] * b[1]
    v = 0
    names = list(MENUS)
    for a in range(len(names)):
        for b in range(a + 1, len(names)):
            i, j = names[a], names[b]
            bi, bj = bundles[i], bundles[j]
            if bi == bj:
                continue
            i_revealed = cost(MENUS[i], bj) <= BUDGET   # j's bundle affordable at i
            j_revealed = cost(MENUS[j], bi) <= BUDGET   # i's bundle affordable at j
            if i_revealed and j_revealed:
                v += 1
    return v


valid["warp_violations"] = valid.apply(warp_violations, axis=1)
share_violators = (valid["warp_violations"] > 0).mean()
print(f"\n=== WARP audit ===")
print(f"Share of agents with >= 1 violation: {share_violators:.3f}")
print(valid.groupby(["persona", "framing"])["warp_violations"].mean().round(3))
print("\nHuman benchmark: ~10% of subjects violate (Andreoni & Miller "
      "2002). BOTH 0% and 50% are divergences — diagnose, don't celebrate.")


## Robustness — required in every lab

Rerun at least the `baseline × plain` and `coffee_devotee × sale` cells
with:

1. **a paraphrased prompt** (rewrite `build_prompt`'s wording, same
   content);
2. **a second model** (change `MODEL` in the setup cell);
3. **non-round prices** — change `MENUS` to e.g. (3.7, 2.3)-style prices
   (and relax the integer check in `parse_response` to match). If
   consistency collapses when the arithmetic stops being easy, what was
   the consistency made of?

**GARP-binding fifth menu (modification 2 in the handout):** with four
menus, pairwise WARP is nearly the whole story; add a fifth menu whose
prices create a potential three-menu cycle (the notebook's WARP function
only checks pairs — extending it to chains is part of the modification).

**Limits of silicon subjects:** the model has read every consumer-theory
textbook; no budget is real to it; and — this module's own lesson — an
agent that never errs is not a model of an agent who rarely errs. Human
data includes the error process, and anything you'd pilot that depends on
error rates (rationality screens, attrition, comprehension) cannot be
calibrated on silicon subjects (write-up Sections 5–6; PS2 Part C).
